# 과제 후보 실측 — USGS 하천 관측 데이터로 무엇을 풀 수 있나

**목적**: 예측 과제를 고르기 위해 `usgs_water`의 실태를 **실측**한다. 규모·시간 범위·결측·
페어링 가능 행 수를 재고, 마지막 셀에 선정 근거표를 남긴다. 그 표가
[`03-water-discharge-baseline.ipynb`](03-water-discharge-baseline.ipynb)의 입력이다.

🔴 **비-목적**: 여기서 모델을 학습하지 않는다. 노트북은 탐색의 자리이고 결론의 근거로 쓰지 않는다
([`analysis.md`](../docs/conventions/analysis.md) §1).

## 왜 이 데이터셋인가 — 기록해 둔다

이 저장소의 분석 질문 정본은 **MIMIC-IV의 SOFA → Sepsis-3**다([`dataset_schema.md`](../docs/dataset_schema.md)).
그러나 2026-09-16 실측 결과 **`mimiciv`·`eicu`는 네임스페이스만 등록돼 있고 Iceberg 테이블이 0건**이며,
`s3://warehouse/raw/mimiciv/icu/`도 비어 있다(hosp 7개 파일만 존재). 즉 **정의는 있으나 실체가 없다**
— 문서가 "적재한다"고 적은 것을 "적재돼 있다"로 읽으면 안 되는 사례다(코딩 철학 #7).

⇒ **지금 실제로 조회되는 데이터는 `usgs_water`뿐**이라 여기서 시작한다.
MIMIC-IV 데이터가 갖춰지면 그쪽이 정본 질문으로 돌아간다.

## 입력 테이블

| 테이블 | 내용 |
|---|---|
| `iceberg.usgs_water.water_iv_raw` | NWIS instantaneous values — 관측소×시각×항목의 측정값 |
| `iceberg.usgs_water.water_sites` | 관측소 메타(위경도·유역면적·고도·주) |

`iv_stream_echo`·`iv_joined`는 **Flink 스트리밍 PoC의 산출물**이라 분석 입력으로 쓰지 않는다.

## 데이터 출처 · 이용 조건

**출처: U.S. Geological Survey (USGS) National Water Information System.**

이 데이터는 **미국 공공영역(U.S. Public Domain)** 이다 — *"USGS-authored or produced data and
information are considered to be in the U.S. Public Domain."* 재배포 제한이 없어 원천 값과 그
파생 수치를 저장소·공개물에 담을 수 있다. 출처 표기는 USGS가 *"we **ask** that proper credit
be given"* 이라 요청하는 것이고 **`must`가 아니므로 의무는 아니지만**, 권고를 따라 위에 적는다.
판정 근거와 `미확인` 잔여는 [`security.md`](../docs/security.md) **§0-1**에 있다.

🔴 **재식별 축은 성립하지 않는다.** 하천 관측소의 수위·유량은 기관 계측값이고 개인 단위 관측이
아니다. 따라서 **소규모 셀 마스킹을 적용하지 않는다** — 빠뜨린 것이 아니라 **적용 대상이 아니라고
판단한 것**이다(MIMIC-IV로 돌아갈 때는 마스킹이 다시 필수가 된다).

⚠️ `frankfurter_fx`와 **재식별 축에서만 같다.** 재배포 축은 다르다 — 그쪽은 ECB가 `must`로
출처 표기를 요구하고 이쪽은 `ask`다. 두 데이터셋을 "같은 취급"으로 묶어 읽지 않는다.

## 전제

```shell
kubectl scale deploy/spark-connect --replicas=1
kubectl rollout status deploy/spark-connect
kubectl get pods -l spark-role=executor   # 🔴 지금 "떠 있음"을 기록해 둔다
```

🔴 마지막 줄이 중요하다. 다 쓰고 내린 뒤의 "executor 없음"은 **지금 있었다는 기록이 있을 때만**
회수를 뜻한다. 처음부터 없었다면 그 확인은 아무 정보도 아니다.

## 산출 엔진

아래 모든 수치의 산출 엔진은 **Spark Connect (Spark 3.5.x)** 다(`analysis.md` §4).

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse

from dotenv import load_dotenv

# repo 루트의 .env를 읽어 os.environ에 주입한다(노트북 cwd = notebooks/).
# 이미 셸에서 export한 값은 덮지 않는다(load_dotenv 기본 override=False).
ENV_PATH = Path.cwd().parent / ".env"
loaded = load_dotenv(ENV_PATH)
SPARK_REMOTE = os.environ.get("SPARK_REMOTE", "sc://localhost:15002")

# `sc://host:port/;use_ssl=true` 형태라 `;` 로 먼저 자른 뒤 파싱한다
# (그대로 urlparse에 넣으면 포트에 `/;use_ssl=true`가 붙어 파싱이 깨진다).
parsed = urlparse(SPARK_REMOTE.split(";")[0].rstrip("/"))
via_ingress = (parsed.hostname or "localhost") not in ("localhost", "127.0.0.1")

print(f".env      : {ENV_PATH} ({'읽음' if loaded else '없음 — 기본값으로 간다'})")
print(f"접속 대상 : {SPARK_REMOTE}")
print(f"경로      : {'TLS Ingress' if via_ingress else 'port-forward(폴백)'}")
if via_ingress and not os.environ.get("GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"):
    print("🔴 GRPC_DEFAULT_SSL_ROOTS_FILE_PATH 미설정 — 자체서명 CA 검증에 실패한다")

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.remote(SPARK_REMOTE).getOrCreate()
print("Spark", spark.version)

# 🔴 관측 경로 생존 확인 — 이 셀이 아래 모든 "0건"·"결측 많음"의 **유효 조건**이다.
#    assert로 조용한 0을 **에러로 바꾼다** — 0건과 "연결이 다른 카탈로그를 보고 있다"는
#    출력이 똑같이 생겼다.
#
#    ⚠️ 실측 함정 하나(2026-09-16): 존재하지 않는 테이블을 조회하면 Spark가
#    "JDBC catalog is initialized without view support" 라는 **엉뚱한 에러**를 낸다.
#    카탈로그 설정 문제처럼 보이지만 실제 원인은 **테이블 부재**다.
#    ⇒ 에러 메시지가 원인을 가리키지 않는다. 먼저 `show tables`로 실재를 확인하라.
tables = [r[1] for r in spark.sql("show tables in iceberg.usgs_water").collect()]
print("usgs_water 테이블:", tables)
assert "water_iv_raw" in tables, (
    "water_iv_raw가 없다 — 적재가 안 됐거나 다른 카탈로그를 보고 있다"
)

n_probe = spark.sql(
    "select count(*) as n from iceberg.usgs_water.water_iv_raw"
).collect()[0]["n"]
assert n_probe > 0, "water_iv_raw가 0건이다. 아래 모든 수치는 무효이므로 여기서 멈춘다."
print(f"water_iv_raw {n_probe:,} 행 — 관측 경로 생존 확인")

In [ ]:
# 분모와 **시간 범위**를 먼저 고정한다. 시간 범위가 이 데이터셋의 성격을 결정한다.
SCALE_SQL = """
select
    count(*) as n_rows,
    count(distinct site_no) as n_sites,
    count(distinct parameter_cd) as n_params,
    count(distinct date_time) as n_timestamps,
    min(date_time) as t_min,
    max(date_time) as t_max
from iceberg.usgs_water.water_iv_raw
"""
scale = spark.sql(SCALE_SQL).toPandas().iloc[0]
display(scale.to_frame("값"))

N_ROWS = int(scale["n_rows"])
N_SITES = int(scale["n_sites"])
span_h = (scale["t_max"] - scale["t_min"]).total_seconds() / 3600.0

print(f"관측 {N_ROWS:,}행 / 관측소 {N_SITES}개 / 항목 {int(scale['n_params'])}종")
print(f"시간 범위: {scale['t_min']} ~ {scale['t_max']}  (**{span_h:.1f}시간**)")
print()
print("🔴 시간 범위를 먼저 본 이유 — 이 값이 과제 후보를 **먼저 잘라낸다**.")
if span_h < 24:
    print(f"   {span_h:.1f}시간짜리 스냅샷이다. 일·계절 주기가 한 번도 돌지 않았으므로")
    print("   **시계열 예측(다음 시점 수위)은 여기서 성립하지 않는다.**")
    print("   데이터가 적은 것이 아니라 **그 질문에 답할 수 있는 축이 없는** 것이다.")

In [ ]:
# 측정 항목(parameter_cd)이 무엇인지 — USGS 코드는 그 자체로 의미가 있다.
PARAM_SQL = """
select
    parameter_cd,
    unit_code,
    count(*) as n,
    count(distinct site_no) as sites,
    sum(case when try_cast(value as double) is null then 1 else 0 end) as not_numeric
from iceberg.usgs_water.water_iv_raw
group by 1, 2
order by n desc
"""
params = spark.sql(PARAM_SQL).toPandas()
PARAM_NAMES = {"00060": "유량 discharge", "00065": "수위 gage height", "00010": "수온"}
params["항목"] = params["parameter_cd"].map(PARAM_NAMES).fillna("기타")
display(params)

# `value`는 string 컬럼이다 — 숫자로 못 바꾸는 값이 섞이면 조용히 NULL이 된다.
bad = int(params["not_numeric"].sum())
print(f"숫자 변환 실패: {bad:,}건")
if bad:
    print("   USGS는 결빙(Ice)·배수영향(Bkw) 같은 품질 코드를 value에 넣기도 한다 —")
    print("   그런 행은 try_cast에서 NULL이 되므로 아래 집계에서 제외된다.")
else:
    print("   전부 수치다. 다만 이것은 '값이 있다'이지 '값이 맞다'가 아니다.")

## 후보 ⓐ 수위 → 유량 (rating curve)

수문학에서 **rating curve**라 부르는 관계다. 관측소마다 하천 단면이 달라
수위(gage height, `00065`)와 유량(discharge, `00060`)의 관계가 다르고, 대략 `Q = a(H - h₀)^b` 꼴을 띤다.

**이 과제가 성립하려면 같은 관측소·같은 시각에 두 항목이 함께 있어야 한다.** 그 페어 수를 센다.

🔴 **누수 주의 — 분할 단위는 행이 아니라 관측소다.** 같은 관측소의 다른 시각 관측이
train과 test에 갈라져 들어가면 모델은 **그 관측소의 곡선을 외운다**. 지표는 좋아지고
새 관측소에 대한 일반화는 0이 된다. (MIMIC-IV에서 환자 단위로 나누는 것과 같은 구조다.)

In [ ]:
# 같은 site_no·date_time에서 수위와 유량을 한 행으로 편다(피벗).
PAIR_SQL = """
with pivoted as (
    select
        site_no,
        date_time,
        max(case when parameter_cd = '00065'
                 then try_cast(value as double) end) as gage_ft,
        max(case when parameter_cd = '00060'
                 then try_cast(value as double) end) as flow_cfs
    from iceberg.usgs_water.water_iv_raw
    group by site_no, date_time
)
select
    count(*) as n_site_time,
    sum(case when gage_ft is not null and flow_cfs is not null
             then 1 else 0 end) as n_paired,
    count(distinct site_no) as n_sites,
    sum(case when gage_ft is not null and flow_cfs is not null
             and flow_cfs > 0 then 1 else 0 end) as n_positive_flow
from pivoted
"""
pair = spark.sql(PAIR_SQL).toPandas().iloc[0]

N_PAIRED = int(pair["n_paired"])
n_site_time = int(pair["n_site_time"])
n_pair_sites = int(pair["n_sites"])
paired_pct = 100.0 * N_PAIRED / n_site_time

print(f"관측소-시각 조합    : {n_site_time:,}")
print(f"수위+유량 동시 관측 : {N_PAIRED:,}  ({paired_pct:.1f}%)")
print(f"관측소 수          : {n_pair_sites}")
print(f"유량 > 0 인 페어    : {int(pair['n_positive_flow']):,}")
print()
print("🔴 학습 행 수는 페어 수이지만,")
print("   **독립 표본 수는 관측소 수에 가깝다**.")
print(f"   {span_h:.1f}시간 안에서 같은 관측소의 값은 거의 변하지 않기 때문이다.")
print(f"   즉 유효 표본은 {N_PAIRED:,}이 아니라 {n_pair_sites}에 가깝다 —")
print("   이것이 이 데이터의 진짜 상한이고, 모델 성능은 이 사실과 함께 읽어야 한다.")

In [ ]:
# 관측소 안에서 값이 실제로 얼마나 움직이는가 — 위 주장(유효 표본 ≈ 관측소 수)의 근거다.
VARIATION_SQL = """
with pivoted as (
    select
        site_no,
        date_time,
        max(case when parameter_cd = '00065'
                 then try_cast(value as double) end) as gage_ft
    from iceberg.usgs_water.water_iv_raw
    group by site_no, date_time
),

per_site as (
    select
        site_no,
        count(*) as n_obs,
        max(gage_ft) - min(gage_ft) as range_ft,
        avg(gage_ft) as mean_ft
    from pivoted
    where gage_ft is not null
    group by site_no
    having count(*) > 1
)

select
    count(*) as n_sites,
    percentile_approx(n_obs, 0.5) as obs_p50,
    percentile_approx(range_ft, array(0.5, 0.9, 0.99)) as range_pctl,
    sum(case when range_ft = 0 then 1 else 0 end) as n_flat
from per_site
"""
var = spark.sql(VARIATION_SQL).toPandas().iloc[0]

print(f"관측이 2건 이상인 관측소: {int(var['n_sites'])}개")
print(f"관측소당 관측 수 중앙값 : {var['obs_p50']}")
print("관측소 내 수위 변동폭(ft) 분위수")
for lab, v in zip(["p50", "p90", "p99"], var["range_pctl"], strict=True):
    print(f"  {lab} : {v:.3f} ft")
print(f"변동이 전혀 없는 관측소  : {int(var['n_flat'])}개")
print()
print("→ 변동폭 중앙값이 0에 가까우면 관측소 내 행들은 **사실상 중복**이다.")
print("  행 수를 표본 수로 읽으면 신뢰구간이 실제보다 좁아 보인다(과신).")

## 피처 후보 — 관측소 메타

수위만으로는 관측소마다 다른 단면을 설명할 수 없다. `water_sites`의 지리·지형 변수가
그 차이를 메울 수 있는지 본다.

- `drain_area_va` — **유역면적**. 물리적으로 유량과 가장 직접 연결되는 변수다.
- `alt_va` — 고도, `dec_lat_va`/`dec_long_va` — 위경도, `state_cd` — 주

이 컬럼들은 전부 **string**으로 적재돼 있어 숫자 변환 가능 비율을 함께 센다.

In [ ]:
# 관측 데이터에 실제로 등장하는 관측소로 좁혀서 메타 결측을 센다.
# (사이트 테이블 797개 중 대부분은 이번 수집에 안 잡힌 관측소다 —
#  분모를 틀리면 결측률이 부풀려진다.)
META_SQL = """
with observed as (
    select distinct site_no from iceberg.usgs_water.water_iv_raw
)
select
    count(*) as n_sites,
    sum(case when try_cast(s.drain_area_va as double) is null
             then 1 else 0 end) as drain_missing,
    sum(case when try_cast(s.alt_va as double) is null
             then 1 else 0 end) as alt_missing,
    sum(case when try_cast(s.dec_lat_va as double) is null
             then 1 else 0 end) as lat_missing,
    count(distinct s.state_cd) as n_states
from observed as o
left join iceberg.usgs_water.water_sites as s on o.site_no = s.site_no
"""
meta = spark.sql(META_SQL).toPandas().iloc[0]

n_meta = int(meta["n_sites"])
print(f"관측에 등장한 관측소: {n_meta}개")
print("🔴 분모는 사이트 테이블 전체가 아니라 이 수다 —")
print("   안 쓰는 관측소의 결측은 셀 이유가 없다.")
print()
for col, label in (
    ("drain_missing", "유역면적"),
    ("alt_missing", "고도"),
    ("lat_missing", "위도"),
):
    n_miss = int(meta[col])
    print(f"  {label:<8} 결측 {n_miss:>4} / {n_meta}  ({100.0 * n_miss / n_meta:.1f}%)")
print(f"  주(state) 종류: {int(meta['n_states'])}")
print()
print("결측률이 높은 피처는 버리는 것이 아니라 **결측 표시자와 함께** 넣는다")
print("— '유역면적이 기재되지 않은 관측소'라는 사실 자체가 정보일 수 있다.")

## 후보 ⓑ·ⓒ — 왜 기각하는가

기각도 근거를 남긴다. 나중에 "왜 안 했나"를 다시 묻지 않기 위해서다.

| 후보 | 기각 사유 |
|---|---|
| ⓑ **다음 시점 수위 예측**(시계열) | 위 §규모에서 실측한 **시간 범위**가 결정적이다. 일주기도 한 번 돌지 않아 추세·계절성을 학습할 축이 없다. 관측소당 관측 수도 한 자릿수~십수 개다 |
| ⓒ **이상치/홍수 경보 분류** | 라벨이 없다. 임계값을 우리가 정하면 그 임계값을 다시 예측하는 **순환**이 된다(MIMIC-IV에서 `sepsis3`를 SOFA로 예측하는 것과 같은 함정). 외부 홍수 기준(NWS flood stage)을 붙이면 성립하나 **그 데이터가 저장소에 없다** |

⇒ 남는 것은 ⓐ **수위 → 유량 회귀**다. 물리적으로 실재하는 관계이고, 라벨을 우리가 만들지 않으며,
관측소 단위 분할로 **일반화 여부를 정직하게 물을 수 있다**.

In [ ]:
import pandas as pd

# 선정 근거표 — 위 셀들의 실측값을 그대로 모은다(손으로 옮겨 적지 않는다).
verdict = pd.DataFrame(
    [
        {
            "후보": "ⓐ 수위→유량 회귀",
            "학습 행": f"{N_PAIRED:,}",
            "유효 표본": f"≈{n_pair_sites} (관측소)",
            "라벨 출처": "관측값(00060)",
            "누수 위험": "관측소 단위 분할로 통제",
            "판정": "✅ 채택",
        },
        {
            "후보": "ⓑ 다음 시점 수위",
            "학습 행": f"{N_ROWS:,}",
            "유효 표본": f"{span_h:.1f}h 범위",
            "라벨 출처": "자기 시계열",
            "누수 위험": "—",
            "판정": "❌ 시간축 부족",
        },
        {
            "후보": "ⓒ 홍수 경보 분류",
            "학습 행": "—",
            "유효 표본": "—",
            "라벨 출처": "**없음**",
            "누수 위험": "임계값 자작 시 순환",
            "판정": "❌ 라벨 부재",
        },
    ]
)
display(verdict)

print("판정 기준(수치를 보기 전에 정한 것):")
print("  1. 라벨을 우리가 만들지 않을 것")
print("     — 만들면 그 정의를 되예측하게 된다")
print("  2. 분할 단위를 분리할 수 있을 것")
print("     — 일반화를 물을 수 없으면 지표가 무의미하다")
print("  3. 질문에 답할 축이 데이터에 있을 것")
print("     — 시간축이 없으면 시계열 질문은 성립하지 않는다")
print()
print(f"⇒ 선정: **수위 → 유량 회귀** ({N_PAIRED:,}행 / 관측소 {n_pair_sites}개)")
print("   다음: 03-water-discharge-baseline.ipynb")

In [ ]:
spark.stop()
print("세션 종료")
print()
print("🔴 컴퓨트 회수 — 터미널에서 실행한다.")
print("   Spark Connect는 클러스터의 유일한 상주 컴퓨트다.")
print("   kubectl scale deploy/spark-connect --replicas=0")
print("   kubectl get pods -l spark-role=executor   # 비어야 한다")
print()
print("   client mode라 executor가 driver와 함께 상주한다. 남아 있으면")
print("   에러도 알림도 없이 1 CPU를 계속 점유한다. 그리고 이 '비었다'는")
print("   기동 시 '떠 있었다'를 기록했을 때만 회수를 뜻한다(첫 셀 전제).")